In [1]:
import torch
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
tqdm.pandas()
from transformers import pipeline, AutoTokenizer,AutoModelForCausalLM
from datasets import load_dataset
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler
import os
import tarfile
import pickle
import json
import matplotlib.pyplot as plt
import torch
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import wandb
import sys
sys.path.append("./docker-python/patches")
#from kaggle_secrets import UserSecretsClient
#user_secrets = UserSecretsClient()
#wandb_api = user_secrets.get_secret("wandb_api")
#wandb.login(key=wandb_api)

In [2]:
config = PPOConfig(
    model_name="lvwerra/gpt2-imdb",
    learning_rate=1.41e-5,
    device="cuda"
    #log_with="wandb",
)

sent_kwargs = {"top_k": None, "function_to_apply": "none", "batch_size": 16}

TypeError: PPOConfig.__init__() got an unexpected keyword argument 'device'

In [3]:
def build_dataset(
    config,
    dataset_name="stanfordnlp/imdb",
    input_min_text_length=2,
    input_max_text_length=8,
):
    """
    Build dataset for training. This builds the dataset from `load_dataset`, one should
    customize this function to train the model on its own dataset.

    Args:
        dataset_name (`str`):
            The name of the dataset to be loaded.

    Returns:
        dataloader (`torch.utils.data.DataLoader`):
            The dataloader for the dataset.
    """
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    tokenizer.pad_token = tokenizer.eos_token
    # load imdb with datasets
    ds = load_dataset(dataset_name, split="train")
    ds = ds.rename_columns({"text": "review"})
    ds = ds.filter(lambda x: len(x["review"]) > 200, batched=False)

    input_size = LengthSampler(input_min_text_length, input_max_text_length)

    def tokenize(sample):
        sample["input_ids"] = tokenizer.encode(sample["review"])[: input_size()]
        sample["query"] = tokenizer.decode(sample["input_ids"])
        return sample

    ds = ds.map(tokenize, batched=False)
    ds.set_format(type="torch")
    return ds

def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

In [4]:
dataset = build_dataset(config, input_min_text_length=4, input_max_text_length=12)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 4713d9dc-0bb2-497f-86a7-dd2bf23458be)')' thrown while requesting HEAD https://huggingface.co/datasets/stanfordnlp/imdb/resolve/main/README.md
Retrying in 1s [Retry 1/5].


Map:   0%|          | 0/24895 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


In [23]:
dataset[0]

{'review': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far 

In [6]:
model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
tokenizer.pad_token = tokenizer.eos_token
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)

In [7]:
ppo_trainer = PPOTrainer(
    config, model, ref_model, tokenizer, dataset=dataset, data_collator=collator
)

In [8]:
device = ppo_trainer.accelerator.device
print(f"Using device: {device}")

Using device: cuda


In [9]:
if ppo_trainer.accelerator.num_processes == 1:
    device = 0 if torch.cuda.is_available() else "cpu"  # to avoid a `pipeline` bug
    print(f"Using device: {device}")
sentiment_pipe = pipeline(
    "sentiment-analysis", model="lvwerra/distilbert-imdb", device=device
)

Using device: 0


In [10]:
text = "this movie was really bad!!"
print(sentiment_pipe(text, **sent_kwargs))

text = "this movie was really good!!"
print(sentiment_pipe(text, **sent_kwargs))

[{'label': 'NEGATIVE', 'score': 2.335048198699951}, {'label': 'POSITIVE', 'score': -2.726576089859009}]
[{'label': 'POSITIVE', 'score': 2.557039737701416}, {'label': 'NEGATIVE', 'score': -2.2947897911071777}]


In [11]:
gen_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}

In [12]:
output_min_length = 32
output_max_length = 64
output_length_sampler = LengthSampler(output_min_length, output_max_length)

generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}

In [13]:
class CFG:
    is_training = True

In [41]:
x = [batch for batch in ppo_trainer.dataloader]

In [54]:
print(x[2]["label"][0])
print(len(x[0]["input_ids"][0]))
print(x[0]["query"])

tensor(0, device='cuda:0')
14
['Pay no attention to the comments behind the curtain! The majority of people', 'This movie was terrible. The', 'Carnosaur 3 is bad... awfully bad. Bad to', "I'll be honest,I finally checked this movie not because of", 'I remember seeing promos for', 'This is the most disturbing film I have ever seen.', 'Can only be described as awful. It is bad to start', 'Cornel Wilde and three dumbbells search for sunk', "I knew it wasn't gunna work out between me", 'I liked Batman: Dead End.', 'Halloween is not only the godfather of all sl', "I haven't yet read Kurt Vonnegut's Mother Night (though", 'Carl Brashear (', 'I had to give this film a 10 simply', "Not one of Keaton's best efforts, this was perhaps a", 'Whoopi was the only reason I watched the Oscars that year.', "I know it's a Power-R", 'In Stand By Me, Vern and Teddy discuss who', 'In sum, overlong and filled with more subplots than sw', "I'm an incorrigible skeptic and agnostic and was thus", 'You just got 

In [64]:
gen_len = output_length_sampler()
query = x[0]["input_ids"][0]

print(f"gen_len - {gen_len}")
generation_kwargs["max_new_tokens"] = gen_len
print(generation_kwargs)
query_response = ppo_trainer.generate(query, **generation_kwargs).squeeze()
print(query_response.shape)
response_len = len(query_response) - len(query)
print(f"query len - {len(query)}")
print(f"response_len - {response_len}")
print(len(query_response[-response_len:]))
response = tokenizer.decode(query_response[-response_len:])
print(f"query : {x[0]['query'][0]}")
print(f"response - {response}")

gen_len - 83
{'min_length': -1, 'top_k': 0.0, 'top_p': 1.0, 'do_sample': True, 'pad_token_id': 50256, 'max_new_tokens': 83}
torch.Size([97])
query len - 14
response_len - 83
83
query : Pay no attention to the comments behind the curtain! The majority of people
response -  say "what's that....ah, no comments, one big sigh, huh" (Corbin Elisabeth, MacArthur)? And I can't think of a specific example from Australian cinema of stilted characters in dialogue.<br /><br />I'm starting further; I could have said: it's all one movie. I could have walked away from that over-redactedness. But..


In [68]:
gen_len = output_length_sampler()
query = x[0]["input_ids"][0]

print(f"gen_len - {gen_len}")
generation_kwargs["max_new_tokens"] = gen_len
print(generation_kwargs)
query_response = ppo_trainer.generate(query, **generation_kwargs).squeeze()
print(query_response.shape)
response_len = len(query_response) - len(query)
print(f"query len - {len(query)}")
print(f"response_len - {response_len}")
print(len(query_response[-response_len:]))
response = tokenizer.decode(query_response[-response_len:])
print(f"query : {x[0]['query'][0]}")
print(f"response - {response}")

full_text = x[0]['query'][0] + response
print(f"full text - {full_text}")
pipe_outputs = sentiment_pipe(full_text, **sent_kwargs)
print(pipe_outputs)

gen_len - 87
{'min_length': -1, 'top_k': 0.0, 'top_p': 1.0, 'do_sample': True, 'pad_token_id': 50256, 'max_new_tokens': 87}
torch.Size([101])
query len - 14
response_len - 87
87
query : Pay no attention to the comments behind the curtain! The majority of people
response -  make a great Mesopotamian snack at some point, but this rogue reindeer is sweet and sexy. Truly, I must applaud every single creature on Bridezilla and Red Dwarf for being awesome.<br /><br />I've lived in England for over 20 years and it's well known to see movies in those exotic cities! I know a long again they are possible considering the status of the "reindeer" that
full text - Pay no attention to the comments behind the curtain! The majority of people make a great Mesopotamian snack at some point, but this rogue reindeer is sweet and sexy. Truly, I must applaud every single creature on Bridezilla and Red Dwarf for being awesome.<br /><br />I've lived in England for over 20 years and it's well known to see movie

In [ ]:
if CFG.is_training:
    for epoch, batch in enumerate(tqdm(ppo_trainer.dataloader)):
        query_tensors = batch["input_ids"]
        print(f"Batch size: {len(query_tensors)}")

        #### Get response from gpt2
        response_tensors = []
        for query in query_tensors:
            #print(f"len query - {len(query)}")
            gen_len = output_length_sampler()
            #print(f"gen_len - {gen_len}")
            generation_kwargs["max_new_tokens"] = gen_len
            query_response = ppo_trainer.generate(query, **generation_kwargs).squeeze()
            #print(query_response.shape)
            response_len = len(query_response) - len(query)
            #print(f"response_len - {response_len}")
            response_tensors.append(query_response[-response_len:])
        batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

        #### Compute sentiment score
        texts = [q + r for q, r in zip(batch["query"], batch["response"])]
        print(f"number of texts for sentiment analysis: {len(texts)}")
        pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
        positive_scores = [
            item["score"]
            for output in pipe_outputs
            for item in output
            if item["label"] == "POSITIVE"
        ]
        print(f"number of positive scores: {len(positive_scores)}")
        rewards = [torch.tensor(score) for score in positive_scores]
        print(f"number of rewards: {len(rewards)}")

        #### Run PPO step
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        ppo_trainer.log_stats(stats, batch, rewards)

  0%|          | 0/194 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Batch size: 128
number of texts for sentiment analysis: 128
number of positive scores: 128
number of rewards: 128


  1%|          | 1/194 [10:27<33:38:52, 627.63s/it]

Batch size: 128
number of texts for sentiment analysis: 128
number of positive scores: 128
number of rewards: 128


  1%|          | 2/194 [17:30<27:02:16, 506.96s/it]

Batch size: 128
number of texts for sentiment analysis: 128
number of positive scores: 128
number of rewards: 128


  2%|▏         | 3/194 [33:36<38:02:15, 716.94s/it]

Batch size: 128
number of texts for sentiment analysis: 128
number of positive scores: 128
number of rewards: 128


In [2]:
import pandas as pd

In [15]:
# create a dummy dataframe

df = pd.DataFrame({
    "epoch": [1, 2, 3, 4, 5],
    "reward": [1.2, 2.3, 3.1, 4.0, 5.5]
})
df

,epoch,reward
0,1,1.2
1,2,2.3
2,3,3.1
3,4,4.0
4,5,5.5


In [16]:
for i, row in df.iterrows():
    print(i)
    df.loc[i, 'index'] = int(row['epoch'] - 1)

0
1
2
3
4


In [17]:
df

,epoch,reward,index
0,1,1.2,0.0
1,2,2.3,1.0
2,3,3.1,2.0
3,4,4.0,3.0
4,5,5.5,4.0
